In [2]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

from pathlib import Path

gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("GPUs:", gpus)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
base_dir = Path("../dataset/pseudo_dataset")

IMG_SIZE = 256

print(base_dir)

../dataset/pseudo_dataset


In [4]:
def load_dataset(split):

    image_dir = base_dir / split / "images"
    mask_dir = base_dir / split / "masks"

    X = []
    Y = []

    for image_path in sorted(image_dir.glob("*")):

        mask_path = mask_dir / f"{image_path.stem}_mask.png"

        if not mask_path.exists():
            continue

        image = cv2.imread(str(image_path))

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        image = cv2.resize(
            image,
            (IMG_SIZE, IMG_SIZE)
        )

        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE
        )

        mask = cv2.resize(
            mask,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_NEAREST
        )

        image = image.astype(np.float32) / 255.0

        mask = (
            mask > 127
        ).astype(np.float32)

        mask = np.expand_dims(
            mask,
            axis=-1
        )

        X.append(image)
        Y.append(mask)

    return np.array(X), np.array(Y)

In [5]:
X_train, Y_train = load_dataset("train")

X_val, Y_val = load_dataset("val")

X_test, Y_test = load_dataset("test")

print("Train:", X_train.shape, Y_train.shape)
print("Val  :", X_val.shape, Y_val.shape)
print("Test :", X_test.shape, Y_test.shape)

Train: (203, 256, 256, 3) (203, 256, 256, 1)
Val  : (43, 256, 256, 3) (43, 256, 256, 1)
Test : (44, 256, 256, 3) (44, 256, 256, 1)


In [6]:
print("Mask min:", Y_train.min())
print("Mask max:", Y_train.max())
print("Unique mask values:", np.unique(Y_train))

Mask min: 0.0
Mask max: 1.0
Unique mask values: [0. 1.]


In [7]:
import albumentations as A

train_aug = A.Compose([

    A.HorizontalFlip(p=0.5),

    A.VerticalFlip(p=0.2),

    A.Rotate(
        limit=15,
        p=0.5
    ),

    A.Affine(
        scale=(0.9, 1.1),
        translate_percent=(-0.05, 0.05),
        rotate=(-10, 10),
        p=0.4
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.3
    )
])

In [8]:
X_train_aug = []
Y_train_aug = []

for image, mask in zip(
    X_train,
    Y_train
):

    # Original
    X_train_aug.append(image)
    Y_train_aug.append(mask)

    # Two augmented versions
    for _ in range(2):

        augmented = train_aug(
            image=(image * 255).astype(np.uint8),
            mask=mask.squeeze().astype(np.uint8)
        )

        aug_image = (
            augmented["image"].astype(np.float32)
            / 255.0
        )

        aug_mask = (
            augmented["mask"] > 0
        ).astype(np.float32)

        aug_mask = np.expand_dims(
            aug_mask,
            axis=-1
        )

        X_train_aug.append(aug_image)
        Y_train_aug.append(aug_mask)

X_train_aug = np.array(X_train_aug)
Y_train_aug = np.array(Y_train_aug)

print(
    "Original:",
    X_train.shape,
    Y_train.shape
)

print(
    "Augmented:",
    X_train_aug.shape,
    Y_train_aug.shape
)

Original: (203, 256, 256, 3) (203, 256, 256, 1)
Augmented: (609, 256, 256, 3) (609, 256, 256, 1)


In [9]:
zero_count = np.sum(
    np.sum(
        Y_train_aug,
        axis=(1, 2, 3)
    ) == 0
)

print("Total augmented:", len(Y_train_aug))
print("Zero masks:", zero_count)
print(
    "Percentage:",
    zero_count / len(Y_train_aug) * 100
)

print("Mask min:", Y_train_aug.min())
print("Mask max:", Y_train_aug.max())
print("Unique:", np.unique(Y_train_aug))

Total augmented: 609
Zero masks: 0
Percentage: 0.0
Mask min: 0.0
Mask max: 1.0
Unique: [0. 1.]


In [10]:
model = tf.keras.models.load_model(
    "../models/unet_dataset1_bce_dice_aug_best.keras",
    compile=False
)

print("Dataset-1 baseline model loaded!")

2026-08-23 13:04:29.531952: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-23 13:04:29.538639: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-23 13:04:29.540683: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Dataset-1 baseline model loaded!


In [11]:
def dice_loss(
    y_true,
    y_pred,
    smooth=1e-6
):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(
        y_true_f * y_pred_f
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        tf.reduce_sum(y_true_f)
        + tf.reduce_sum(y_pred_f)
        + smooth
    )

    return 1.0 - dice


def bce_dice_loss(
    y_true,
    y_pred
):

    bce = tf.keras.losses.binary_crossentropy(
        y_true,
        y_pred
    )

    bce = tf.reduce_mean(bce)

    dice = dice_loss(
        y_true,
        y_pred
    )

    return bce + dice

In [12]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=bce_dice_loss,
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        )
    ]
)

In [13]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "../models/unet_dataset2_refined.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

In [14]:
history = model.fit(
    X_train_aug,
    Y_train_aug,
    validation_data=(X_val, Y_val),
    epochs=10,
    batch_size=2,
    callbacks=[checkpoint]
)

Epoch 1/10


2026-08-23 13:05:58.693128: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2026-08-23 13:06:01.551654: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f432b6cb4b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-08-23 13:06:01.551677: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-08-23 13:06:01.554780: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787470561.610879   19110 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-08-23 13:06:01.783664: W external/local_tsl/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.14GiB with freed_by_count=0. The caller

305/305 [==============================] - ETA: 0s - loss: 0.2194 - accuracy: 0.9850
Epoch 1: val_loss improved from inf to 0.17039, saving model to ../models/unet_dataset2_refined.keras
305/305 [==============================] - 66s 171ms/step - loss: 0.2194 - accuracy: 0.9850 - val_loss: 0.1704 - val_accuracy: 0.9880
Epoch 2/10
305/305 [==============================] - ETA: 0s - loss: 0.1758 - accuracy: 0.9876
Epoch 2: val_loss improved from 0.17039 to 0.15477, saving model to ../models/unet_dataset2_refined.keras
305/305 [==============================] - 46s 151ms/step - loss: 0.1758 - accuracy: 0.9876 - val_loss: 0.1548 - val_accuracy: 0.9894
Epoch 3/10
305/305 [==============================] - ETA: 0s - loss: 0.1611 - accuracy: 0.9890
Epoch 3: val_loss did not improve from 0.15477
305/305 [==============================] - 47s 153ms/step - loss: 0.1611 - accuracy: 0.9890 - val_loss: 0.1549 - val_accuracy: 0.9897
Epoch 4/10
305/305 [==============================] - ETA: 0s - lo

In [15]:
model = tf.keras.models.load_model(
    "../models/unet_dataset2_refined.keras",
    compile=False
)

print("Best Dataset-2 model loaded!")

Best Dataset-2 model loaded!


In [16]:
predictions = model.predict(
    X_test,
    batch_size=1,
    verbose=1
)

pred_masks = (
    predictions >= 0.5
).astype(np.float32)

print("Predictions shape:", predictions.shape)

44/44 [==============================] - 1s 22ms/step
Predictions shape: (44, 256, 256, 1)


In [17]:
def dice_score(y_true, y_pred, smooth=1e-6):

    y_true = y_true.astype(np.float32)
    y_pred = y_pred.astype(np.float32)

    intersection = np.sum(y_true * y_pred)

    return (
        2.0 * intersection + smooth
    ) / (
        np.sum(y_true)
        + np.sum(y_pred)
        + smooth
    )


def iou_score(y_true, y_pred, smooth=1e-6):

    y_true = y_true.astype(np.float32)
    y_pred = y_pred.astype(np.float32)

    intersection = np.sum(y_true * y_pred)

    union = (
        np.sum(y_true)
        + np.sum(y_pred)
        - intersection
    )

    return (
        intersection + smooth
    ) / (
        union + smooth
    )

In [18]:
dice_scores = []
iou_scores = []

for i in range(len(Y_test)):

    true_mask = Y_test[i].squeeze()
    pred_mask = pred_masks[i].squeeze()

    dice_scores.append(
        dice_score(
            true_mask,
            pred_mask
        )
    )

    iou_scores.append(
        iou_score(
            true_mask,
            pred_mask
        )
    )

dice_scores = np.array(dice_scores)
iou_scores = np.array(iou_scores)

print("Dataset-2 Refined Model")
print("-----------------------")

print("Test images:", len(Y_test))

print("Mean Dice:", dice_scores.mean())
print("Mean IoU:", iou_scores.mean())

print("Median Dice:", np.median(dice_scores))
print("Median IoU:", np.median(iou_scores))

Dataset-2 Refined Model
-----------------------
Test images: 44
Mean Dice: 0.9072357986239719
Mean IoU: 0.8435017030936612
Median Dice: 0.9399739825692981
Median IoU: 0.8867462591708035


In [19]:
baseline_model = tf.keras.models.load_model(
    "../models/unet_dataset1_bce_dice_aug_best.keras",
    compile=False
)

In [20]:
baseline_predictions = baseline_model.predict(
    X_test,
    batch_size=1,
    verbose=1
)

baseline_pred_masks = (
    baseline_predictions >= 0.5
).astype(np.float32)

44/44 [==============================] - 1s 22ms/step


In [21]:
baseline_dice = []
baseline_iou = []

for i in range(len(Y_test)):

    true_mask = Y_test[i].squeeze()
    pred_mask = baseline_pred_masks[i].squeeze()

    baseline_dice.append(
        dice_score(true_mask, pred_mask)
    )

    baseline_iou.append(
        iou_score(true_mask, pred_mask)
    )

baseline_dice = np.array(baseline_dice)
baseline_iou = np.array(baseline_iou)

print("Dataset-1 Baseline → Dataset-2 Test")
print("------------------------------------")
print("Mean Dice:", baseline_dice.mean())
print("Mean IoU:", baseline_iou.mean())
print("Median Dice:", np.median(baseline_dice))
print("Median IoU:", np.median(baseline_iou))

Dataset-1 Baseline → Dataset-2 Test
------------------------------------
Mean Dice: 0.8610613879160618
Mean IoU: 0.7817056291917559
Median Dice: 0.9066610622904755
Median IoU: 0.8293084851683827


In [22]:
# Testing the 522 test images on the refined model 

In [23]:
import tensorflow as tf
import numpy as np
import cv2
from pathlib import Path

datasetPath = Path("../dataset/data_wound_seg_256")

IMG_SIZE = 256

test_images_path = sorted(
    (datasetPath / "test_images").glob("*.png")
)

test_masks_path = sorted(
    (datasetPath / "test_masks").glob("*.png")
)

print("Images:", len(test_images_path))
print("Masks :", len(test_masks_path))

Images: 552
Masks : 552


In [24]:
X_test_ds1 = []
Y_test_ds1 = []

for image_path, mask_path in zip(
    test_images_path,
    test_masks_path
):

    image = cv2.imread(str(image_path))

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    image = cv2.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    mask = cv2.resize(
        mask,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    image = image.astype(
        np.float32
    ) / 255.0

    mask = (
        mask > 127
    ).astype(np.float32)

    mask = np.expand_dims(
        mask,
        axis=-1
    )

    X_test_ds1.append(image)
    Y_test_ds1.append(mask)

X_test_ds1 = np.array(X_test_ds1)
Y_test_ds1 = np.array(Y_test_ds1)

print("X:", X_test_ds1.shape)
print("Y:", Y_test_ds1.shape)

X: (552, 256, 256, 3)
Y: (552, 256, 256, 1)


In [25]:
model = tf.keras.models.load_model(
    "../models/unet_dataset2_refined.keras",
    compile=False
)

print("Dataset-2 refined model loaded!")

Dataset-2 refined model loaded!


In [26]:
predictions = model.predict(
    X_test_ds1,
    batch_size=1,
    verbose=1
)

pred_masks = (
    predictions >= 0.5
).astype(np.float32)

print(
    "Predictions:",
    predictions.shape
)

2026-08-23 13:21:44.666585: W external/local_tsl/tsl/framework/bfc_allocator.cc:485] Allocator (GPU_0_bfc) ran out of memory trying to allocate 16.00MiB (rounded to 16777216)requested by op U_Net_BCE_Dice/activation_1/Relu
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2026-08-23 13:21:44.666653: I external/local_tsl/tsl/framework/bfc_allocator.cc:1039] BFCAllocator dump for GPU_0_bfc
2026-08-23 13:21:44.666669: I external/local_tsl/tsl/framework/bfc_allocator.cc:1046] Bin (256): 	Total Chunks: 98, Chunks in use: 98. 24.5KiB allocated for chunks. 24.5KiB in use in bin. 9.3KiB client-requested in use in bin.
2026-08-23 13:21:44.666681: I external/local_tsl/tsl/framework/bfc_allocator.cc:1046] Bin (512): 	Total Chunks: 30, Chunks in use: 30. 15.8KiB allocated for chunks. 15.8KiB in use in bin. 15.0KiB client-requested in use in bin

ResourceExhaustedError: Graph execution error:

Detected at node U_Net_BCE_Dice/activation_1/Relu defined at (most recent call last):
  File "/home/hosurvijay/miniconda3/envs/tf_gpu/lib/python3.10/runpy.py", line 196, in _run_module_as_main

  File "/home/hosurvijay/miniconda3/envs/tf_gpu/lib/python3.10/runpy.py", line 86, in _run_code

  File "/home/hosurvijay/miniconda3/envs/tf_gpu/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/hosurvijay/miniconda3/envs/tf_gpu/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/hosurvijay/miniconda3/envs/tf_gpu/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/home/hosurvijay/miniconda3/envs/tf_gpu/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/home/hosurvijay/miniconda3/envs/tf_gpu/lib/python3.10/asyncio/base_events.py", line 603, in run_forever

  File "/home/hosurvijay/minicond

In [ ]:
def dice_score(
    y_true,
    y_pred,
    smooth=1e-6
):

    y_true = y_true.astype(np.float32)
    y_pred = y_pred.astype(np.float32)

    intersection = np.sum(
        y_true * y_pred
    )

    return (
        2 * intersection + smooth
    ) / (
        np.sum(y_true)
        + np.sum(y_pred)
        + smooth
    )


def iou_score(
    y_true,
    y_pred,
    smooth=1e-6
):

    y_true = y_true.astype(np.float32)
    y_pred = y_pred.astype(np.float32)

    intersection = np.sum(
        y_true * y_pred
    )

    union = (
        np.sum(y_true)
        + np.sum(y_pred)
        - intersection
    )

    return (
        intersection + smooth
    ) / (
        union + smooth
    )

In [ ]:
dice_scores = []
iou_scores = []

for i in range(len(Y_test_ds1)):

    true_mask = Y_test_ds1[i].squeeze()
    pred_mask = pred_masks[i].squeeze()

    dice_scores.append(
        dice_score(
            true_mask,
            pred_mask
        )
    )

    iou_scores.append(
        iou_score(
            true_mask,
            pred_mask
        )
    )

dice_scores = np.array(dice_scores)
iou_scores = np.array(iou_scores)

print("Dataset-2 Refined Model → Dataset-1 Test")
print("-----------------------------------------")

print(
    "Test images:",
    len(Y_test_ds1)
)

print(
    "Mean Dice:",
    dice_scores.mean()
)

print(
    "Mean IoU:",
    iou_scores.mean()
)

print(
    "Median Dice:",
    np.median(dice_scores)
)

print(
    "Median IoU:",
    np.median(iou_scores)
)